# LangChain Example Selector Base Reference

# `BaseExampleSelector: ABC`

Abstract interface for storing examples and selecting examples to include in prompts.

A concrete subclass must implement `add_example()` and `select_examples()`. The asynchronous methods `aadd_example()` and `aselect_examples()` are inherited wrappers that run the corresponding synchronous methods through `run_in_executor()`.

## Required subclass hooks

### `add_example`

Adds an example to the selector's store.

```python
@abstractmethod
add_example(
    self,
    example: dict[str, str], # Input-variable names mapped to their example values
) -> Any # Implementation-specific return value
```

### `select_examples`

Selects examples based on the supplied input variables.

```python
@abstractmethod
select_examples(
    self,
    input_variables: dict[str, str], # Input-variable names mapped to their current values
) -> list[dict[str, Any]] # Selected examples
```

## Methods

### `aadd_example`

Asynchronously runs `add_example()` through `run_in_executor()`.

```python
async aadd_example(
    self,
    example: dict[str, str], # Input-variable names mapped to their example values
) -> Any # Return value from add_example()
```

### `aselect_examples`

Asynchronously runs `select_examples()` through `run_in_executor()`.

```python
async aselect_examples(
    self,
    input_variables: dict[str, str], # Input-variable names mapped to their current values
) -> list[dict[str, Any]] # Selected examples
```

In [ ]:
from typing import Any # Import Any for the return type
from langchain_core.example_selectors import BaseExampleSelector # Import the abstract selector


class TopicExampleSelector(BaseExampleSelector): # Create a concrete example selector
    def __init__(self) -> None: # Initialize the selector
        self.examples: list[dict[str, str]] = [] # Store all examples

    def add_example(self, example: dict[str, str]) -> None: # Add an example
        self.examples.append(example) # Save the example in the list

    def select_examples(
        self,
        input_variables: dict[str, str], # Receive the current input values
    ) -> list[dict[str, Any]]: # Return matching examples
        topic = input_variables.get("topic", "") # Read the requested topic

        return [ # Return examples with the same topic
            example # Return the matching example
            for example in self.examples # Check every stored example
            if example.get("topic") == topic # Keep examples with a matching topic
        ]


selector = TopicExampleSelector() # Create the selector

selector.add_example( # Add the first example
    {
        "topic": "Python",
        "question": "What is a list?",
        "answer": "A list stores multiple values.",
    }
)

selector.add_example( # Add the second example
    {
        "topic": "SQL",
        "question": "What is GROUP BY?",
        "answer": "GROUP BY creates groups of rows.",
    }
)

selector.add_example( # Add another Python example
    {
        "topic": "Python",
        "question": "What is a tuple?",
        "answer": "A tuple is an immutable collection.",
    }
)

python_examples = selector.select_examples({"topic": "Python"}) # Select examples synchronously

print("Synchronous results:") # Display a heading

for example in python_examples: # Visit each selected example
    print(example) # Display the example

await selector.aadd_example( # Add an example asynchronously in Jupyter
    {
        "topic": "Pandas",
        "question": "What does axis=1 mean?",
        "answer": "It performs the operation across columns for each row.",
    }
)

pandas_examples = await selector.aselect_examples( # Select examples asynchronously
    {"topic": "Pandas"}
)

print("\nAsynchronous results:") # Display another heading

for example in pandas_examples: # Visit each asynchronous result
    print(example) # Display the example